# Attack detection on the Thermostat device (TON_IoT)
**Models:** Decision Tree • Random Forest • XGBoost • LightGBM • CatBoost

Same pipeline, same split rule and same export format as the Weather device, so the aggregation step can read both.

1. EDA  2. Cleaning + 10-second windows  3. Safe hash-block split  4. Training  5. Best-model analysis
6. Block cross-validation  7. Multi-class  8. Ablation + random split demo  9. Calibration, anomaly score, export, **self-check**

**Rules against leakage:** `type` is the label (not a feature), `date`/`time`/`ts` are not features,
the split uses contiguous blocks (not random rows), thresholds are chosen on validation only.

In [ ]:
import os, warnings, hashlib, json
import numpy as np, pandas as pd
import matplotlib
if os.environ.get("HEADLESS"): matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (precision_score, recall_score, f1_score, confusion_matrix,
                             roc_curve, precision_recall_curve, roc_auc_score)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", rc={"axes.grid.axis": "y"})
plt.rcParams["figure.dpi"] = 100
SEED = 42
SOURCE = "thermostat"
RAW_SENSORS = {"current_temperature": "temperature", "thermostat_status": "status"}   # raw column -> short name
SENSORS = list(RAW_SENSORS.values())
DATA_PATH = os.environ.get("DATA_PATH", "/content/IoT_Thermostat.csv")

if not os.path.exists(DATA_PATH):
    from google.colab import files
    DATA_PATH = "/content/" + list(files.upload().keys())[0]

# 1) EDA

In [ ]:
raw = pd.read_csv(DATA_PATH)
print("Shape:", raw.shape)
missing_cols = [c for c in ["date", "time", "label", "type", *RAW_SENSORS] if c not in raw.columns]
assert not missing_cols, f"These columns are missing from the file: {missing_cols}. Found: {list(raw.columns)}"
raw = raw.rename(columns=RAW_SENSORS)

In [ ]:
raw.head()

In [ ]:
print(raw.dtypes, "\n")
print("Missing values:\n", raw.isna().sum(), "\n")
print("Exact duplicates:", raw.duplicated().sum())

In [ ]:
# rows with no date/time can't be placed in a 10-second window (in this file: normal rows with temp=0, status=0)
n_before = len(raw)
dropped = raw[raw["date"].isna() | raw["time"].isna()]
print(f"Dropping {len(dropped)} rows without date/time | their types: {dropped['type'].value_counts().to_dict()}")
raw = raw.dropna(subset=["date", "time"]).reset_index(drop=True)
raw["time"] = raw["time"].astype(str).str.strip()
raw["date"] = raw["date"].astype(str).str.strip()
print("Duplicates after cleaning the time column:", raw.duplicated().sum())
raw[SENSORS].describe().round(2)

In [ ]:
print(raw["label"].value_counts(), "\n")
print(raw["type"].value_counts())
print("\nthermostat_status values:", sorted(raw["status"].dropna().unique())[:10])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
raw["label"].map({0: "Normal", 1: "Attack"}).value_counts().plot.bar(ax=ax[0], color=["#4c72b0", "#c44e52"], rot=0)
ax[0].set_title("Normal vs Attack")
raw["type"].value_counts().plot.bar(ax=ax[1], color="#55a868")
ax[1].set_yscale("log"); ax[1].set_title("Records per type (log scale)")
plt.tight_layout(); plt.show()

In [ ]:
day = pd.to_datetime(raw["date"], format="%d-%b-%y")
att = raw[raw["label"] == 1]
pd.crosstab(day[att.index].dt.strftime("%d-%b"), att["type"]).plot.bar(stacked=True, figsize=(12, 4), colormap="tab10")
plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.title("Attack records per day"); plt.xlabel("Date"); plt.ylabel("Records"); plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.kdeplot(data=raw.sample(min(100_000, len(raw)), random_state=SEED), x="temperature", hue="label", common_norm=False,
            fill=True, palette=["#4c72b0", "#c44e52"], ax=ax[0]); ax[0].set_title("temperature by label")
sns.boxplot(data=raw, x="type", y="temperature", order=raw["type"].value_counts().index, showfliers=False, ax=ax[1])
ax[1].tick_params(axis="x", rotation=45); ax[1].set_title("temperature by type")
plt.tight_layout(); plt.show()

In [ ]:
# thermostat_status is a 0/1 state: show the share of status==1 for every type
st = raw.groupby("type")["status"].mean().sort_values()
st.plot.barh(figsize=(6, 3.5), color="#8172b2"); plt.title("Share of records with thermostat_status = 1"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(4.5, 3.5))
sns.heatmap(raw[SENSORS + ["label"]].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation"); plt.show()

In [ ]:
first = raw.index[raw["label"] == 1][0]
seg = raw.iloc[max(first - 200, 0): first + 1500].reset_index(drop=True)
fig, ax = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
for a, c in zip(ax, SENSORS):
    a.plot(seg[c], lw=0.7, color="gray")
    a.scatter(seg.index[seg.label == 1], seg.loc[seg.label == 1, c], s=4, color="red"); a.set_ylabel(c)
ax[-1].set_xlabel("Row order"); plt.tight_layout(); plt.show()

# 2) Cleaning + 10-second windows

In [ ]:
df = raw.drop_duplicates().reset_index(drop=True)
print(f"Rows: {len(raw)} -> {len(df)} after removing duplicates")

In [ ]:
ts = pd.to_datetime(df["date"] + " " + df["time"], format="%d-%b-%y %H:%M:%S")   # one timestamp per row

In [ ]:
# The file is several time parts stuck together: when time goes backward a new chunk starts.
df["chunk"] = (ts.diff().dt.total_seconds() < 0).cumsum()
df["win"] = ((ts - pd.Timestamp("2019-01-01")).dt.total_seconds() // 10).astype(int)

In [ ]:
g = df.groupby(["chunk", "win"], sort=False)
W = g[SENSORS].agg(["mean", "std", "min", "max"])
W.columns = [f"{a}_{b}" for a, b in W.columns]
W["n_records"] = g.size()

In [ ]:
counts = pd.crosstab([df["chunk"], df["win"]], df["type"]).reindex(W.index)
W["type"] = counts.idxmax(axis=1).values                       # majority vote inside the window
W["is_attack"] = (W["type"] != "normal").astype(int)
W = W.reset_index().fillna(0)

In [ ]:
W["prev_n_records"] = W.groupby("chunk")["n_records"].shift(1).fillna(0)
W["gap_prev"] = W.groupby("chunk")["win"].diff().fillna(0)

In [ ]:
for c in ["temperature_mean", "temperature_std", "status_mean", "n_records"]:
    for k in (1, 2, 3):
        W[f"{c}_lag{k}"] = W.groupby("chunk")[c].shift(k)
    W[f"{c}_roll3"] = W.groupby("chunk")[c].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
    W[f"{c}_chg"] = W[c] - W[f"{c}_roll3"]
W = W.fillna(0)

FEATURES = [c for c in W.columns if c not in ("chunk", "win", "type", "is_attack")]
print(f"Windows: {len(W)} | Attack windows: {W.is_attack.mean():.1%} | Features: {len(FEATURES)}")

In [ ]:
W.head()

In [ ]:
plt.figure(figsize=(9, 4))
sns.histplot(data=W[W.n_records <= 30], x="n_records", hue=W["is_attack"].map({0: "Normal", 1: "Attack"}),
             stat="density", common_norm=False, discrete=True, palette=["#4c72b0", "#c44e52"], element="step")
plt.title("Records per 10-second window"); plt.show()

In [ ]:
run_id = (W["is_attack"] != W["is_attack"].shift()).cumsum()
run_len = W[W.is_attack == 1].groupby(run_id).size()
plt.figure(figsize=(9, 3.5)); plt.hist(run_len.clip(upper=60), bins=30, color="#c44e52")
plt.xlabel("Consecutive attack windows in a row (clipped at 60)"); plt.ylabel("Count of attack runs")
plt.title("How long do attacks last?"); plt.show()
print(f"Median run = {run_len.median():.0f} windows | longest = {run_len.max()} windows")

# 3) Safe split (identical rule on every device)

In [ ]:
BLOCK = 100
SALT = "iot-anomaly"
TRAIN_PCT, VAL_PCT = 70, 15

n = len(W)
all_idx = np.arange(n)

def assign_split(win, bw=BLOCK, salt=SALT):
    block = np.asarray(win) // bw
    bucket = {b: int(hashlib.md5(f"{salt}:{b}".encode()).hexdigest(), 16) % 100 for b in np.unique(block)}
    r = np.array([bucket[b] for b in block])
    return np.where(r < TRAIN_PCT, "train", np.where(r < TRAIN_PCT + VAL_PCT, "val", "test"))

block_id = W["win"].values // BLOCK
split_name = assign_split(W["win"].values)
train, val, test = (np.where(split_name == s)[0] for s in ("train", "val", "test"))

pos = W["win"].values % BLOCK
edge = (pos < 3) | (pos >= BLOCK - 3)            # safety gap: lag / rolling look 3 windows back
train, val, test = [i[~edge[i]] for i in (train, val, test)]

assert not set(block_id[train]) & set(block_id[val])
assert not set(block_id[train]) & set(block_id[test])
assert not set(block_id[val])   & set(block_id[test])

X, y = W[FEATURES], W["is_attack"].values
y_tr, y_va, y_te = y[train], y[val], y[test]

display(pd.DataFrame({"windows": [len(train), len(val), len(test)],
                      "attack %": [y_tr.mean() * 100, y_va.mean() * 100, y_te.mean() * 100]},
                     index=["train", "val", "test"]).round(1))
pd.DataFrame({"train": W["type"].iloc[train].value_counts(), "val": W["type"].iloc[val].value_counts(),
              "test": W["type"].iloc[test].value_counts()}).fillna(0).astype(int)

# 4) Model

In [ ]:
spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)

def make_models(spw):
    return {
        "DecisionTree": DecisionTreeClassifier(max_depth=10, min_samples_leaf=20, class_weight="balanced", random_state=SEED),
        "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=10,
                                               class_weight="balanced_subsample", n_jobs=-1, random_state=SEED),
        "XGBoost": XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, min_child_weight=5,
                                 subsample=0.8, colsample_bytree=0.8, reg_lambda=5, scale_pos_weight=spw,
                                 early_stopping_rounds=50, eval_metric="aucpr", n_jobs=-1, random_state=SEED),
        "LightGBM": LGBMClassifier(n_estimators=1000, learning_rate=0.05, num_leaves=31, min_child_samples=20,
                                   subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=5,
                                   class_weight="balanced", n_jobs=-1, random_state=SEED, verbose=-1),
        "CatBoost": CatBoostClassifier(iterations=600, learning_rate=0.08, depth=6, l2_leaf_reg=5,
                                       auto_class_weights="Balanced", early_stopping_rounds=50,
                                       random_seed=SEED, verbose=0),
    }

def fit(name, model, Xtr, ytr, Xva, yva, sample_weight=None):
    if name == "XGBoost":
        model.fit(Xtr, ytr, sample_weight=sample_weight, eval_set=[(Xva, yva)], verbose=False)
    elif name == "LightGBM":
        model.fit(Xtr, ytr, sample_weight=sample_weight, eval_set=[(Xva, yva)], callbacks=[early_stopping(50, verbose=False)])
    elif name == "CatBoost":
        model.fit(Xtr, ytr, sample_weight=sample_weight, eval_set=(Xva, yva))
    else:
        model.fit(Xtr, ytr, sample_weight=sample_weight)
    return model

def best_threshold(y_true, proba):
    ths = np.linspace(0.05, 0.95, 91)
    return ths[np.argmax([f1_score(y_true, proba >= t) for t in ths])]

In [ ]:
models, probs, rows = {}, {}, []
for name, m in make_models(spw).items():
    m = fit(name, m, X.iloc[train], y_tr, X.iloc[val], y_va)
    p_tr, p_va, p_te = (m.predict_proba(X.iloc[i])[:, 1] for i in (train, val, test))
    th = best_threshold(y_va, p_va)                       # threshold chosen on validation only
    models[name], probs[name] = m, dict(train=p_tr, val=p_va, test=p_te, th=th)
    for split, y_true, p in (("train", y_tr, p_tr), ("test", y_te, p_te)):
        rows.append(dict(model=name, split=split, threshold=round(th, 2),
                         recall=recall_score(y_true, p >= th), precision=precision_score(y_true, p >= th, zero_division=0),
                         f1=f1_score(y_true, p >= th), auc=roc_auc_score(y_true, p)))
results = pd.DataFrame(rows).round(3)
cmp = results.pivot(index="model", columns="split", values=["recall", "precision", "f1", "auc"])
cmp[("f1", "gap")] = cmp[("f1", "train")] - cmp[("f1", "test")]      # overfitting gap
cmp.round(3)

# 5) Best model analysis

In [ ]:
best = max(probs, key=lambda n: f1_score(y_va, probs[n]["val"] >= probs[n]["th"]))   # validation only
print("Best model by validation F1:", best)
pt, th = probs[best]["test"], probs[best]["th"]

In [ ]:
from sklearn.isotonic import IsotonicRegression
calibrator = IsotonicRegression(out_of_bounds="clip").fit(probs[best]["val"], y_va)   # fitted on validation only
p_cal_va = calibrator.predict(probs[best]["val"])
p_cal_te = calibrator.predict(probs[best]["test"])
th_cal = best_threshold(y_va, p_cal_va)
print(f"Calibrated threshold: {th_cal:.2f}")
print(f"Test F1 (calibrated): {f1_score(y_te, p_cal_te >= th_cal):.3f}")

In [ ]:
cm = confusion_matrix(y_te, pt >= th)
plt.figure(figsize=(5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=["Normal", "Attack"], yticklabels=["Normal", "Attack"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"{best} — Confusion Matrix"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
for nm, d in probs.items():
    fpr, tpr, _ = roc_curve(y_te, d["test"]); plt.plot(fpr, tpr, label=nm)
plt.plot([0, 1], [0, 1], "k--", lw=0.7); plt.title("ROC"); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
for nm, d in probs.items():
    p, r, _ = precision_recall_curve(y_te, d["test"]); plt.plot(r, p, label=nm)
plt.title("Precision-Recall"); plt.xlabel("Recall"); plt.ylabel("Precision"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
imp = pd.Series(models[best].feature_importances_, index=FEATURES).sort_values(ascending=False)
imp.head(15)[::-1].plot.barh(figsize=(8, 6), color="#55a868")
plt.title(f"{best} feature importance (top 15)"); plt.xlabel("Importance"); plt.tight_layout(); plt.show()

In [ ]:
ths = np.linspace(0.02, 0.98, 97)
rec = [recall_score(y_te, pt >= t) for t in ths]
pre = [precision_score(y_te, pt >= t, zero_division=1) for t in ths]
f1s = [f1_score(y_te, pt >= t) for t in ths]
plt.figure(figsize=(9, 4))
plt.plot(ths, rec, label="Recall"); plt.plot(ths, pre, label="Precision"); plt.plot(ths, f1s, label="F1", ls="--")
plt.axvline(th, color="gray", ls=":", label=f"chosen on validation ({th:.2f})")
plt.xlabel("Threshold"); plt.legend(); plt.title(f"{best}: threshold trade-off"); plt.show()

pv = probs[best]["val"]; out = []
for target in (0.90, 0.95):
    ok = [t for t in ths if recall_score(y_va, pv >= t) >= target]
    t = max(ok) if ok else ths[0]
    out.append(dict(target_recall=target, threshold=round(t, 2), test_recall=recall_score(y_te, pt >= t),
                    test_precision=precision_score(y_te, pt >= t, zero_division=0), test_f1=f1_score(y_te, pt >= t)))
pd.DataFrame(out).round(3)

# 6) Cross-Validation

In [ ]:
cv_rows = []
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold, (tr_i, te_i) in enumerate(outer.split(all_idx, W["type"].values, groups=block_id)):
    tr_i, te_i = tr_i[~edge[tr_i]], te_i[~edge[te_i]]
    inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    a, b = next(inner.split(tr_i, W["type"].values[tr_i], groups=block_id[tr_i]))
    fit_i, va_i = tr_i[a], tr_i[b]
    spw_k = (y[fit_i] == 0).sum() / max((y[fit_i] == 1).sum(), 1)
    for name, m in make_models(spw_k).items():
        m = fit(name, m, X.iloc[fit_i], y[fit_i], X.iloc[va_i], y[va_i])
        th_k = best_threshold(y[va_i], m.predict_proba(X.iloc[va_i])[:, 1])
        pred = m.predict_proba(X.iloc[te_i])[:, 1] >= th_k
        cv_rows.append(dict(fold=fold, model=name, recall=recall_score(y[te_i], pred),
                            precision=precision_score(y[te_i], pred, zero_division=0), f1=f1_score(y[te_i], pred)))
    print("fold", fold, "done")
cv_df = pd.DataFrame(cv_rows)
cv_summary = cv_df.groupby("model")[["recall", "precision", "f1"]].agg(["mean", "std"]).round(3)
cv_summary

# 7) Multi-Class

In [ ]:
le = LabelEncoder().fit(W["type"]); classes = list(le.classes_)
yc = le.transform(W["type"]); yc_tr, yc_va, yc_te = yc[train], yc[val], yc[test]
sw = compute_sample_weight("balanced", yc_tr)
absent = [classes[k] for k in range(len(classes)) if k not in set(yc_tr)]
assert not absent, f"Classes with no windows in the TRAIN split: {absent} (XGBoost multi-class needs every class in train)"
print("Test support:", dict(pd.Series(W["type"].values[test]).value_counts()))

mc = {
    "DecisionTree": DecisionTreeClassifier(max_depth=10, min_samples_leaf=20, class_weight="balanced", random_state=SEED),
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=10, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED),
    "XGBoost": XGBClassifier(n_estimators=600, learning_rate=0.07, max_depth=5, min_child_weight=3, subsample=0.8, colsample_bytree=0.8,
                             reg_lambda=5, early_stopping_rounds=40, n_jobs=-1, random_state=SEED),
    "LightGBM": LGBMClassifier(n_estimators=600, learning_rate=0.07, num_leaves=31, min_child_samples=15, subsample=0.8, subsample_freq=1,
                               colsample_bytree=0.8, reg_lambda=5, class_weight="balanced", n_jobs=-1, random_state=SEED, verbose=-1),
    "CatBoost": CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, l2_leaf_reg=5, loss_function="MultiClass",
                                   auto_class_weights="Balanced", early_stopping_rounds=40, random_seed=SEED, verbose=0),
}
present = sorted(set(yc_te)); mc_pred, mc_rows = {}, []
for name, m in mc.items():
    m = fit(name, m, X.iloc[train], yc_tr, X.iloc[val], yc_va, sample_weight=sw if name == "XGBoost" else None)
    pred = np.asarray(m.predict(X.iloc[test])).ravel().astype(int); mc_pred[name] = pred
    mc_rows.append(dict(model=name, macro_f1=f1_score(yc_te, pred, labels=present, average="macro", zero_division=0),
                        weighted_f1=f1_score(yc_te, pred, average="weighted", zero_division=0)))
mc_results = pd.DataFrame(mc_rows).set_index("model").round(3)
mc_results

In [ ]:
val_f1 = {n: f1_score(yc_va, np.asarray(m.predict(X.iloc[val])).ravel().astype(int),
                      labels=sorted(set(yc_va)), average="macro") for n, m in mc.items()}
best_mc = max(val_f1, key=val_f1.get); print("Best multi-class model:", best_mc)
pred = mc_pred[best_mc]
cm = confusion_matrix(yc_te, pred, labels=range(len(classes)))[present]
cm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", ax=ax[0], xticklabels=classes, yticklabels=[classes[i] for i in present])
ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("Actual"); ax[0].set_title(f"{best_mc} — Confusion Matrix (row-normalised)")
per_class = pd.DataFrame({m: [(p[yc_te == k] == k).mean() for k in present] for m, p in mc_pred.items()},
                         index=[classes[i] for i in present])
per_class.plot.bar(ax=ax[1], rot=45, colormap="Set2"); ax[1].set_title("Recall per attack type"); ax[1].set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

# 8) Ablation + Random Split

In [ ]:
TIMING = [c for c in FEATURES if "n_records" in c or c == "gap_prev"]
def quick_f1(cols):
    m = fit("XGBoost", make_models(spw)["XGBoost"], X[cols].iloc[train], y_tr, X[cols].iloc[val], y_va)
    th_ = best_threshold(y_va, m.predict_proba(X[cols].iloc[val])[:, 1])
    pr = m.predict_proba(X[cols].iloc[test])[:, 1] >= th_
    return dict(recall=recall_score(y_te, pr), precision=precision_score(y_te, pr, zero_division=0), f1=f1_score(y_te, pr))

abl = pd.DataFrame({"all features": quick_f1(FEATURES),
                    "sensors only (no counts)": quick_f1([c for c in FEATURES if c not in TIMING]),
                    "counts/timing only": quick_f1(TIMING)}).T.round(3)
abl.plot.bar(rot=15, figsize=(9, 4), colormap="Set2"); plt.ylim(0, 1); plt.title("Ablation (XGBoost, Test)"); plt.show()
abl

In [ ]:
a, b = train_test_split(all_idx, test_size=0.2, stratify=y, random_state=SEED)
a, v = train_test_split(a, test_size=0.2, stratify=y[a], random_state=SEED)
rows = []
for name, m in make_models(spw).items():
    m = fit(name, m, X.iloc[a], y[a], X.iloc[v], y[v])
    th_ = best_threshold(y[v], m.predict_proba(X.iloc[v])[:, 1])
    rows.append(dict(model=name, random_split=f1_score(y[b], m.predict_proba(X.iloc[b])[:, 1] >= th_),
                     block_split=f1_score(y_te, probs[name]["test"] >= probs[name]["th"])))
cmp = pd.DataFrame(rows).set_index("model").round(3)
cmp.plot.bar(rot=0, figsize=(9, 4), color=["#c44e52", "#4c72b0"])
plt.ylim(0, 1); plt.ylabel("Test F1"); plt.title("Random split (leaky) vs Block split (honest)"); plt.show()
print(cmp)

In [ ]:
os.makedirs("results", exist_ok=True)
results.to_csv("results/binary_results.csv"); mc_results.to_csv("results/multiclass_results.csv")
abl.to_csv("results/ablation.csv"); cv_summary.to_csv("results/cv_summary.csv")
per_class.to_csv("results/recall_per_attack_type.csv"); cmp.to_csv("results/random_vs_block.csv")
print("Saved in ./results")

# 9) Anomaly score + export

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EmpiricalCovariance
from sklearn.ensemble import IsolationForest

norm_tr = train[y[train] == 0]

def to_percentile(raw_score):
    ref = np.sort(raw_score[norm_tr])
    return np.searchsorted(ref, raw_score) / len(ref)

sc = StandardScaler().fit(X.iloc[norm_tr])
raw_z = np.abs(sc.transform(X)).mean(axis=1)
cov = EmpiricalCovariance().fit(X.iloc[norm_tr])
raw_m = cov.mahalanobis(X)
iso = IsolationForest(n_estimators=200, random_state=SEED, n_jobs=-1).fit(X.iloc[norm_tr])
raw_i = -iso.score_samples(X)
scores = {"Z-score": to_percentile(raw_z), "Mahalanobis": to_percentile(raw_m), "IsolationForest": to_percentile(raw_i)}
print("done:", list(scores))

In [ ]:
from sklearn.neural_network import MLPRegressor
sc_ae = StandardScaler().fit(X.iloc[norm_tr])
Xn = sc_ae.transform(X.iloc[norm_tr])
ae = MLPRegressor(hidden_layer_sizes=(32, 16, 8, 16, 32), activation="relu", solver="adam",
                  max_iter=200, early_stopping=True, n_iter_no_change=10, random_state=SEED)
ae.fit(Xn, Xn)
X_all = sc_ae.transform(X)
raw_ae = ((X_all - ae.predict(X_all)) ** 2).mean(axis=1)
scores["Autoencoder"] = to_percentile(raw_ae)
print("done:", list(scores))

In [ ]:
from sklearn.metrics import average_precision_score
ths = np.linspace(0.01, 0.99, 99)
th_an = {nm: ths[np.argmax([f1_score(y[val], s[val] >= t) for t in ths])] for nm, s in scores.items()}
rows = []
for nm, s in scores.items():
    for split_nm, ii in (("val", val), ("test", test)):
        pred = s[ii] >= th_an[nm]
        rows.append(dict(model=nm, split=split_nm, threshold=round(th_an[nm], 2), auc=roc_auc_score(y[ii], s[ii]),
                         avg_precision=average_precision_score(y[ii], s[ii]), baseline_ap=y[ii].mean(),
                         recall=recall_score(y[ii], pred), precision=precision_score(y[ii], pred, zero_division=0),
                         f1=f1_score(y[ii], pred)))
anom_results = pd.DataFrame(rows).set_index(["model", "split"]).round(3)
anom_results

In [ ]:
best_an = anom_results.xs("val", level="split")["f1"].idxmax()
anomaly = scores[best_an]
print("Anomaly method:", best_an)
if best_an == "Mahalanobis":
    anomaly_model = {"method": best_an, "model": cov, "reference": np.sort(raw_m[norm_tr])}
elif best_an == "Autoencoder":
    anomaly_model = {"method": best_an, "model": ae, "scaler": sc_ae, "reference": np.sort(raw_ae[norm_tr])}
elif best_an == "IsolationForest":
    anomaly_model = {"method": best_an, "model": iso, "reference": np.sort(raw_i[norm_tr])}
else:
    anomaly_model = {"method": best_an, "scaler": sc, "reference": np.sort(raw_z[norm_tr])}

In [ ]:
import joblib
OUT = os.environ.get("EXPORT_DIR", "export")
os.makedirs(OUT, exist_ok=True); os.makedirs("results", exist_ok=True)

mc_best = mc[best_mc]
th_final = float(th_cal)

pa = calibrator.predict(models[best].predict_proba(X)[:, 1])
pred_type = le.inverse_transform(np.asarray(mc_best.predict(X)).ravel().astype(int))
is_attack_pred = pa >= th_final
attack_type = np.where(is_attack_pred, np.where(pred_type != "normal", pred_type, "unknown"), "none")

split = np.full(len(W), "purged", dtype=object)
for idx, key in ((train, "train"), (val, "val"), (test, "test")):
    split[idx] = key

events = pd.DataFrame({
    "source": SOURCE,
    "window_start": pd.Timestamp("2019-01-01") + pd.to_timedelta(W["win"] * 10, unit="s"),
    "prediction": np.where(is_attack_pred, "Attack", "Normal"),
    "attack_prob": pa,
    "attack_type": attack_type,
    "anomaly_score": anomaly,
    "n_records": W["n_records"].values,
    "true_label": W["is_attack"].values,
    "split": split,
})
events.insert(0, "event_id", range(len(events)))
events.to_csv(f"{OUT}/{SOURCE}_events.csv", index=False)
anom_results.to_csv("results/anomaly_results.csv")

p = np.full(len(W), np.nan)
for idx, key in ((train, "train"), (val, "val"), (test, "test")):
    p[idx] = probs[best][key]
export_df = pd.DataFrame({"source": SOURCE, "chunk": W["chunk"], "win": W["win"], "split": split,
                          "p_attack": p, "y_true": W["is_attack"], "true_type": W["type"],
                          "pred_type": pred_type, "n_records": W["n_records"]})
export_df[export_df["split"] != "purged"].to_csv(f"{OUT}/{SOURCE}_predictions.csv", index=False)

joblib.dump({"binary": models[best], "calibrator": calibrator, "multiclass": mc_best,
             "label_encoder": le, "anomaly": anomaly_model}, f"{OUT}/{SOURCE}_model.joblib")

cfg = {"source": SOURCE, "binary_model": best, "multiclass_model": best_mc, "calibrated": True,
       "threshold": th_final, "anomaly_method": best_an, "features": FEATURES, "classes": classes,
       "window_seconds": 10, "epoch": "2019-01-01", "block_windows": BLOCK,
       "split_rule": {"method": "md5 hash of block number", "salt": SALT, "train_pct": TRAIN_PCT, "val_pct": VAL_PCT},
       "val_f1": float(f1_score(y_va, p_cal_va >= th_final)),
       "test_f1": float(f1_score(y_te, p_cal_te >= th_final))}
json.dump(cfg, open(f"{OUT}/{SOURCE}_config.json", "w"), indent=2)
print("Exported to", OUT, "| val F1:", round(cfg["val_f1"], 4), "| test F1:", round(cfg["test_f1"], 4))

# 10) Self-check: reload the exported model and make sure the numbers are consistent

In [ ]:
bundle = joblib.load(f"{OUT}/{SOURCE}_model.joblib")
ev = pd.read_csv(f"{OUT}/{SOURCE}_events.csv")
pa2 = bundle["calibrator"].predict(bundle["binary"].predict_proba(X)[:, 1])
checks = {
    "reloaded model reproduces attack_prob exactly": np.allclose(pa2, ev["attack_prob"].values, atol=1e-9),
    "reloaded multiclass reproduces attack_type": (np.where(pa2 >= th_final, np.where(le.inverse_transform(np.asarray(bundle["multiclass"].predict(X)).ravel().astype(int)) != "normal",
                                                    le.inverse_transform(np.asarray(bundle["multiclass"].predict(X)).ravel().astype(int)), "unknown"), "none") == ev["attack_type"].values).all(),
    "no NaN / inf in features": bool(np.isfinite(X.to_numpy(float)).all()),
    "no NaN / inf in events": bool(np.isfinite(ev[["attack_prob", "anomaly_score", "n_records"]].to_numpy(float)).all()),
    "probabilities inside [0,1]": bool(ev["attack_prob"].between(0, 1).all() and ev["anomaly_score"].between(0, 1).all()),
    "val AND test each contain both Normal and Attack windows": all(set(ev.loc[ev.split == s_, "true_label"]) == {0, 1} for s_ in ("val", "test")),
    "model is not degenerate (does not predict one class for every test window)": 0 < (ev.loc[ev.split == "test", "prediction"] == "Attack").mean() < 1,
    "test F1 from events file == config": abs(f1_score(ev[ev.split == "test"].true_label, ev[ev.split == "test"].prediction.eq("Attack")) - cfg["test_f1"]) < 1e-9,
}
for k, v in checks.items(): print("PASS" if v else "FAIL", k)
assert all(checks.values()), "Some checks failed - do not use these exports"